In [ ]:
import pandas as pd
import random
from transformers import pipeline
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Labeling (Menggunakan model roBERTa)

In [ ]:
# 1. Inisialisasi Model Indo-RoBERTa
# Menggunakan model publik yang sudah dilatih untuk sentimen bahasa Indonesia
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

print("Memuat model Indo-RoBERTa... (Ini mungkin memakan waktu beberapa saat)")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    truncation=True, # Mencegah error jika ada teks yang terlalu panjang
    max_length=512
)

# 2. Simulasi Data Mentah (Ganti dengan data Anda)
# Disini saya membuat 25 data simulasi agar kita bisa mengambil 20 data acak nantinya.
try:
    df = pd.read_csv('/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/komentar_youtube_clean.csv')
    data_mentah = df['clean_text'].tolist()
    print(f"Data berhasil diakses. Total {len(data_mentah)} komentar bersih.")
except FileNotFoundError:
    print("Error: file CSV tidak ditemukan.")
    data_mentah = [] # Set to empty list or handle error as appropriate
except KeyError:
    print("Error: 'clean_text' kolom tidak ditemukan di file CSV tersebut.")
    data_mentah = []
except Exception as e:
    print(f"Error saat membaca file CSV: {e}")
    data_mentah = []

Memuat model Indo-RoBERTa... (Ini mungkin memakan waktu beberapa saat)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Data berhasil diakses. Total 10990 komentar bersih.


In [ ]:
# 3. Proses Auto-Labeling
print("Memulai proses auto-labeling...")
hasil_prediksi = sentiment_pipeline(data_mentah)

# Menggabungkan teks asli dengan hasil label ke dalam struktur List
data_berlabel = []
for teks, hasil in zip(data_mentah, hasil_prediksi):
    data_berlabel.append({
        "Teks": teks,
        "Label": hasil['label'].capitalize(), # Mengubah 'positive' jadi 'Positive'
        "Skor Kepercayaan": round(hasil['score'], 3) # Membulatkan skor probabilitas
    })

# 4. Konversi ke Pandas DataFrame agar rapi
df_hasil = pd.DataFrame(data_berlabel)

# (Opsional) Simpan semua data ke CSV sebelum di-sample
# df_hasil.to_csv('semua_data_berlabel.csv', index=False)

Memulai proses auto-labeling...


In [ ]:
# 5. Evaluasi: Tampilkan 20 Data Secara Random
print("\n" + "="*50)
print("HASIL EVALUASI: 20 DATA RANDOM")
print("="*50)

# Menggunakan pandas .sample() untuk mengambil data acak
# random_state=None agar hasil acaknya berubah-ubah setiap kali dijalankan
df_evaluasi = df_hasil.sample(n=20, random_state=None)

# Menampilkan dataframe dengan format tabel rapi
display(df_evaluasi) # Gunakan print(df_evaluasi.to_string()) jika tidak jalan di Jupyter/Colab


HASIL EVALUASI: 20 DATA RANDOM


,Teks,Label,Skor Kepercayaan
7976,kagak tahu malu amerika mengatur wilayah orang...,Negative,0.999
5627,bbm naik alhamdulillah saya ngarepin,Positive,0.903
253,asal as bisa mengendalikan anjingnya,Negative,0.550
1333,semua gara gara sih wowok,Negative,0.998
3964,komentar proff uki malah tidak jos memalukan k...,Negative,0.999
5454,jalesveva jaya mahe ala iran di laut kita kara...,Negative,0.963
4101,pengamat boncos,Negative,0.971
5536,oh bangsa cerdas yang suka bun h anak gaz itu ...,Negative,0.795
6506,kata kata hari ini jangan pernah percaya denga...,Negative,0.998
2756,sudah benar jadi negara netral malah ikut kubu as,Negative,0.992


In [ ]:
print("\n" + "="*50)
print("PERSEBARAN LABELING DATA")
print("="*50)

# Menghitung persebaran label
label_distribution = df_hasil['Label'].value_counts()

# Menghitung persentase
total_labels = len(df_hasil)
label_percentages = (label_distribution / total_labels) * 100

# Menggabungkan hasil ke dalam DataFrame untuk tampilan yang rapi
df_label_distribution = pd.DataFrame({
    'Jumlah': label_distribution,
    'Persentase': label_percentages.round(2).astype(str) + '%'
})

print(df_label_distribution)


PERSEBARAN LABELING DATA
          Jumlah Persentase
Label                      
Negative    8533     77.64%
Neutral     1495      13.6%
Positive     962      8.75%


In [ ]:
df_hasil.to_csv('/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/Komentar_Youtube_bersih_berlabel.csv', index=False)